# Bucketing

In [0]:
from pyspark.sql.functions import rand

# przykładowy df
df = spark.range(0, 10000).withColumn("id_mod_10", (rand() * 10).cast("int"))

In [0]:
# zapis danych z użyciem partitionBy

df.write.mode("overwrite") \
  .partitionBy("id_mod_10") \
  .parquet("/tmp/partitioned_data")

In [0]:
# zapis danych z użyciem bucketing

spark.sql("DROP TABLE IF EXISTS bucketed_table")

(df.write
   .mode("overwrite")
   .format("parquet")
   .bucketBy(10, "id_mod_10")
   .sortBy("id_mod_10")
   .saveAsTable("bucketed_parquet_table"))

In [0]:
# porównanie struktury plików

display(dbutils.fs.ls("/tmp/partitioned_data"))
display(dbutils.fs.ls("dbfs:/user/hive/warehouse/bucketed_parquet_table"))

path,name,size,modificationTime
dbfs:/tmp/partitioned_data/_SUCCESS,_SUCCESS,0,1748369303000
dbfs:/tmp/partitioned_data/id_mod_10=0/,id_mod_10=0/,0,0
dbfs:/tmp/partitioned_data/id_mod_10=1/,id_mod_10=1/,0,0
dbfs:/tmp/partitioned_data/id_mod_10=2/,id_mod_10=2/,0,0
dbfs:/tmp/partitioned_data/id_mod_10=3/,id_mod_10=3/,0,0
dbfs:/tmp/partitioned_data/id_mod_10=4/,id_mod_10=4/,0,0
dbfs:/tmp/partitioned_data/id_mod_10=5/,id_mod_10=5/,0,0
dbfs:/tmp/partitioned_data/id_mod_10=6/,id_mod_10=6/,0,0
dbfs:/tmp/partitioned_data/id_mod_10=7/,id_mod_10=7/,0,0
dbfs:/tmp/partitioned_data/id_mod_10=8/,id_mod_10=8/,0,0


path,name,size,modificationTime
dbfs:/user/hive/warehouse/bucketed_parquet_table/_SUCCESS,_SUCCESS,0,1748369575000
dbfs:/user/hive/warehouse/bucketed_parquet_table/_committed_2764066708831392257,_committed_2764066708831392257,5890,1748369575000
dbfs:/user/hive/warehouse/bucketed_parquet_table/_started_2764066708831392257,_started_2764066708831392257,0,1748369569000
dbfs:/user/hive/warehouse/bucketed_parquet_table/part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-1_00001.c000.snappy.parquet,part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-1_00001.c000.snappy.parquet,1928,1748369570000
dbfs:/user/hive/warehouse/bucketed_parquet_table/part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-2_00002.c000.snappy.parquet,part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-2_00002.c000.snappy.parquet,1374,1748369571000
dbfs:/user/hive/warehouse/bucketed_parquet_table/part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-3_00003.c000.snappy.parquet,part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-3_00003.c000.snappy.parquet,1918,1748369572000
dbfs:/user/hive/warehouse/bucketed_parquet_table/part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-4_00004.c000.snappy.parquet,part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-4_00004.c000.snappy.parquet,1332,1748369573000
dbfs:/user/hive/warehouse/bucketed_parquet_table/part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-5_00006.c000.snappy.parquet,part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-5_00006.c000.snappy.parquet,1417,1748369573000
dbfs:/user/hive/warehouse/bucketed_parquet_table/part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-6_00007.c000.snappy.parquet,part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-6_00007.c000.snappy.parquet,1397,1748369574000
dbfs:/user/hive/warehouse/bucketed_parquet_table/part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-7_00009.c000.snappy.parquet,part-00000-tid-2764066708831392257-8f726bc5-1c57-467e-ac95-7700a321d62d-8-7_00009.c000.snappy.parquet,1894,1748369575000


# Wnioski
- partitionBy lepiej nadaje się, gdy filtrujemy po konkretnej wartości
- bucketing jest efektywniejsze przy joinach na dużych zbiorach danych, bo Spark może pomijać etap shuffle, jeśli buckety są zgodne.
- pliki na dysku różnią się: partitioning tworzy wiele folderów, bucketing – pliki bucketów w jednym folderze.